## Homework: Crypto Spread Trading

## Data Preprocessing

In [1]:
import numpy as np
import polars as pl
import pandas as pd

In [2]:
def load_and_preprocess_single_date(
    exchange_name: str,
    date_str: str,
) -> pd.DataFrame:
    """
    Load and preprocess trade data for a single exchange and given date.

    Loads a parquet file, filters for trades (`rec_type == "T"`), converts relevant
    columns to appropriate types, sets the timestamp as the index, resamples
    to 1-second frequency (taking the last trade in each second), and returns
    only the `price` and `qty` columns as a pandas DataFrame.

    Args:
        exchange_name (str): Name of the exchange (e.g., 'binance', 'OKX', 'Coinbase').
        date_str (str): Date string in the format 'YYYYMMDD'.

    Returns:
        pd.DataFrame: Preprocessed DataFrame with 'price' and 'qty' indexed by timestamp.
    """
    parquet_path = f"data/ETH-USDT/{exchange_name}/{date_str}.parquet"
    df_polars = pl.read_parquet(parquet_path)
    df_polars = df_polars.filter(pl.col("rec_type") == "T")
    df = df_polars.to_pandas()
    df["price"] = df["price"].astype(float)
    df["qty"] = df["qty"].astype(float)
    df = df.set_index("ts")
    df = df.resample("1s").last()
    df = df[["price", "qty"]]
    return df

In [3]:
def load_multiple_exchanges_dates(exchange_names, dates):
    """
    Load and preprocess trade data for multiple exchanges and dates.

    Returns:
        dict: Keys are f"{exchange}_{date}" and values are preprocessed DataFrames.
    """
    return {
        f"{name}_{date}": load_and_preprocess_single_date(name, date)
        for name in exchange_names
        for date in dates
    }

exchange_names = ['Binance', 'Coinbase', 'OKX']
dates = ['20250524', '20250525']
exchange_date_dfs = load_multiple_exchanges_dates(exchange_names, dates)

In [6]:
def concatenate_exchange_dataframes(
    exchange_date_dfs: dict,
    exchanges: list = None
) -> dict:
    """
    Concatenate trade DataFrames for each exchange across all available dates.

    Args:
        exchange_date_dfs (dict): A dictionary with keys in the format 'Exchange_Date'
                                  and values as pandas DataFrames.
        exchanges (list, optional): List of exchange names to process. Defaults to
                                    ['Binance', 'Coinbase', 'OKX'].

    Returns:
        dict: Dictionary with keys as exchange names and values as concatenated
              and time-sorted DataFrames per exchange.
    """
    if exchanges is None:
        exchanges = ['Binance', 'Coinbase', 'OKX']

    concatenated_dfs = {}
    for exchange in exchanges:
        # Collect all DataFrames belonging to this exchange across all dates
        dfs = [
            df for key, df in exchange_date_dfs.items()
            if key.startswith(f"{exchange}_")
        ]
        if dfs:
            # Concatenate along the time index and sort chronologically
            concatenated_dfs[exchange] = pd.concat(dfs).sort_index()

    return concatenated_dfs

# Concatenate data for all exchanges, then assign each exchange's DataFrame to its own variable
exchange_concat_dfs = concatenate_exchange_dataframes(exchange_date_dfs)
binance = exchange_concat_dfs.get('Binance')
coinbase = exchange_concat_dfs.get('Coinbase')
okx = exchange_concat_dfs.get('OKX')

# Combine price columns from each exchange into a single DataFrame
price_panel = pd.concat([binance['price'].rename('binance_price'),
                        coinbase['price'].rename('coinbase_price'),
                        okx['price'].rename('okx_price')], axis=1)

# Forward fill the missing values and drop NaNs
price_panel = price_panel.ffill().dropna()

In [9]:
price_panel

,binance_price,coinbase_price,okx_price
ts,,,
2025-05-24 00:00:06,2525.53,2525.57,2525.60
2025-05-24 00:00:07,2525.31,2525.75,2525.53
2025-05-24 00:00:08,2525.84,2525.75,2526.00
2025-05-24 00:00:09,2525.79,2525.75,2526.10
2025-05-24 00:00:10,2525.80,2525.75,2525.90
...,...,...,...
2025-05-25 23:59:55,2551.22,2551.62,2551.41
2025-05-25 23:59:56,2551.23,2551.62,2551.20
2025-05-25 23:59:57,2551.22,2551.62,2551.20
